# 实验二：自适应视频流与 QoE 优化
## 从规则策略到强化学习 · Adaptive Bitrate Streaming (ABR)

**课程**：未来媒体互联网（Future Media & Internet） &nbsp;|&nbsp; **预计时长**：~12 分钟  
**运行环境**：Kaggle Notebook（CPU） &nbsp;|&nbsp; **无需 GPU** &nbsp;|&nbsp; **Internet Off**

---

### 一句话问题

> **当网络带宽不断变化时，播放器应该怎样选择下一段视频的码率，才能既清晰、又少卡顿、还不要频繁跳画质？**


## 实验概述

自适应码率（Adaptive Bitrate Streaming, ABR）是 DASH / HLS 等流媒体系统的核心机制。播放器不是一次下载整部视频，而是按 **Segment（视频分段）**逐段下载；每下载下一段之前，都要重新选择码率。

本实验把这个过程简化为一个可运行的序贯决策问题：

**网络带宽波动 → 选择码率 → Segment 下载时间 → Buffer 增减 → Rebuffer → QoE**

我们对比三种策略：

1. **Fixed High**：始终选择 10 Mbps；
2. **Buffer Heuristic**：根据缓冲区水位使用人工规则；
3. **Tabular Q-Learning**：从大量仿真带宽轨迹中学习状态—动作价值，并在测试阶段使用学到的策略。

与原版相比，本实验采用真正的 **2 秒 Segment-level 模拟器**，并把“卡顿次数”改为更合理的 **Rebuffering Time（秒）**。


## 学习目标

完成本实验后，你应该能够：

1. 解释 **Bitrate、Bandwidth、Buffer、Rebuffering** 之间的关系；
2. 理解为什么“始终选择最高码率”通常不是最佳用户体验；
3. 比较固定策略、缓冲区启发式策略和 Q-Learning 策略的差异；
4. 解释强化学习中的 **State → Action → Reward → Q Update**；
5. 区分 **训练阶段的探索（exploration）** 与 **部署/评测阶段的确定性策略**；
6. 用 QoE 指标分析“画质、卡顿、画质波动”三者之间的权衡。


## 背景与实验设计

### 1. Segment-level ABR

本实验每个视频 Segment 的播放时长为 **2 秒**。若选择码率 \(R\) Mbps，则一个 Segment 的数据量为：

\[
\text{segment size}=R\times 2
\]

若当前实际吞吐量为 \(B\) Mbps，则下载时间为：

\[
\text{download time}=\frac{R\times 2}{B}
\]

下载期间视频仍在播放，因此 Buffer 会被消耗。如果下载时间超过当前 Buffer，就产生 **Rebuffering（重新缓冲）**。

---

### 2. QoE：不是只看画质

本实验采用一个简化、可解释的 QoE：

\[
QoE =
\overline{R}
-\alpha \cdot \frac{T_{rebuffer}}{N}
-\beta \cdot \frac{\sum |R_t-R_{t-1}|}{N}
\]

其中：

- \(\overline{R}\)：平均码率，越高越好；
- \(T_{rebuffer}\)：累计重新缓冲时间，越少越好；
- \(|R_t-R_{t-1}|\)：相邻 Segment 的码率变化幅度，越小越稳定；
- \(N\)：Segment 数量。

本实验取 `alpha = 12.0`、`beta = 0.8`，强调“卡顿比单纯画质下降更伤体验”。

---

### 3. 强化学习状态

为了让“画质变化惩罚”真正进入决策，本实验状态包含：

- Buffer：5 档；
- 上一次观测到的 Bandwidth：3 档；
- 上一个 Segment 的 Bitrate：3 档。

因此一共只有：

**5 × 3 × 3 = 45 个状态，3 个动作。**

这仍然是一张很小的 Q-table，适合课堂理解。


## 运行环境

| 项目 | 设置 |
|---|---|
| 平台 | Kaggle Notebook |
| 计算资源 | CPU |
| GPU | 不需要 |
| Internet | Off |
| 外部数据集 | 不需要 |
| 依赖 | NumPy、Pandas、Matplotlib |

直接 **Run All** 即可运行。


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

QUALITY_LEVELS = np.array([2.0, 5.0, 10.0])  # Mbps
CHUNK_DURATION = 2.0                         # seconds
MAX_BUFFER = 20.0
INITIAL_BUFFER = 6.0
INITIAL_BW_EST = 5.0

REBUFFER_PENALTY = 12.0
SMOOTHNESS_PENALTY = 0.8

print("Environment ready")
print(f"Bitrate ladder: {QUALITY_LEVELS.tolist()} Mbps")
print(f"Segment duration: {CHUNK_DURATION:.1f} s")


## 步骤一：生成测试带宽轨迹

我们生成 60 个 Segment 对应的无线网络吞吐量，共代表 **120 秒视频**。轨迹同时包含：

- 慢速周期变化；
- 较快波动；
- 平滑随机噪声；
- 少量突发带宽下降。

注意：播放器在真实系统中通常只能根据**过去已经完成的下载**估计下一段可用吞吐量。本实验中的 Q-Learning 也遵守这一点：决策时使用的是“上一段观测到的带宽”，而不是偷看未来。


In [2]:
def generate_bandwidth_trace(n_segments=60, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    t = np.arange(n_segments)

    phase1 = rng.uniform(0, 2 * np.pi)
    phase2 = rng.uniform(0, 2 * np.pi)

    base = (
        7.0
        + 3.0 * np.sin(2 * np.pi * t / 24 + phase1)
        + 1.8 * np.sin(2 * np.pi * t / 9 + phase2)
    )

    noise = rng.normal(0, 1.0, n_segments)
    for i in range(1, n_segments):
        noise[i] = 0.65 * noise[i - 1] + 0.35 * noise[i]

    trace = np.clip(base + noise, 0.8, 14.0)

    # Add a few short bandwidth drops
    for _ in range(rng.integers(1, 4)):
        start = int(rng.integers(5, n_segments - 5))
        duration = int(rng.integers(2, 5))
        trace[start:start + duration] *= rng.uniform(0.45, 0.70)

    return np.clip(trace, 0.6, 14.0)


test_bandwidth = generate_bandwidth_trace(
    n_segments=60,
    rng=np.random.default_rng(42)
)

video_time = np.arange(len(test_bandwidth)) * CHUNK_DURATION

plt.figure(figsize=(12, 4))
plt.plot(video_time, test_bandwidth, linewidth=2, label="Available bandwidth")
for bitrate in QUALITY_LEVELS:
    plt.axhline(bitrate, linestyle="--", alpha=0.6, label=f"{bitrate:g} Mbps bitrate")
plt.xlabel("Video time (s)")
plt.ylabel("Mbps")
plt.title("Synthetic Bandwidth Trace and Bitrate Ladder")
plt.grid(alpha=0.2)
plt.legend(ncol=2)
plt.show()

print(
    f"Bandwidth range: {test_bandwidth.min():.2f}–{test_bandwidth.max():.2f} Mbps | "
    f"mean: {test_bandwidth.mean():.2f} Mbps"
)


## 步骤二：建立真正的 Segment-level 播放模拟器

对每个 Segment：

1. 策略根据当前状态选择 2 / 5 / 10 Mbps；
2. 根据实际带宽计算该 Segment 的下载时间；
3. 下载期间 Buffer 被持续消耗；
4. 如果 Buffer 不够，就累计 **Rebuffering Time**；
5. 下载完成后，Buffer 增加 2 秒视频；
6. 将本次实际吞吐量作为下一次决策的带宽估计。

三种策略共享完全相同的环境，区别只在“如何选下一档码率”。


In [3]:
def environment_step(buffer, prev_action, action, actual_bw):
    bitrate = QUALITY_LEVELS[action]
    segment_size = bitrate * CHUNK_DURATION
    download_time = segment_size / actual_bw

    rebuffer = max(download_time - buffer, 0.0)
    new_buffer = max(buffer - download_time, 0.0) + CHUNK_DURATION
    new_buffer = min(new_buffer, MAX_BUFFER)

    variation = abs(bitrate - QUALITY_LEVELS[prev_action])

    reward = (
        bitrate
        - REBUFFER_PENALTY * rebuffer
        - SMOOTHNESS_PENALTY * variation
    )

    return new_buffer, rebuffer, variation, reward, download_time


def state_index(buffer, bw_estimate, prev_action):
    buffer_state = min(int(buffer // 4), 4)  # 0..4
    if bw_estimate < 4:
        bw_state = 0
    elif bw_estimate < 8:
        bw_state = 1
    else:
        bw_state = 2

    return (buffer_state * 3 + bw_state) * 3 + prev_action


def strategy_fixed(buffer, prev_action, bw_estimate, segment_id):
    return 2  # always 10 Mbps


def strategy_heuristic(buffer, prev_action, bw_estimate, segment_id):
    # Pure buffer-based rule
    if buffer < 4:
        return 0
    if buffer < 8:
        return 1
    if buffer > 14:
        return 2
    return prev_action


def simulate(trace, policy):
    buffer = INITIAL_BUFFER
    prev_action = 1
    bw_estimate = INITIAL_BW_EST

    rows = []
    total_rebuffer = 0.0
    total_variation = 0.0
    bitrates = []

    for segment_id, actual_bw in enumerate(trace):
        action = int(policy(buffer, prev_action, bw_estimate, segment_id))

        new_buffer, rebuffer, variation, reward, download_time = environment_step(
            buffer, prev_action, action, actual_bw
        )

        bitrate = float(QUALITY_LEVELS[action])
        rows.append({
            "segment": segment_id,
            "video_time": segment_id * CHUNK_DURATION,
            "bandwidth": actual_bw,
            "bw_estimate": bw_estimate,
            "buffer_before": buffer,
            "bitrate": bitrate,
            "download_time": download_time,
            "rebuffer": rebuffer,
            "buffer_after": new_buffer,
        })

        total_rebuffer += rebuffer
        total_variation += variation
        bitrates.append(bitrate)

        buffer = new_buffer
        prev_action = action
        bw_estimate = float(actual_bw)

    avg_bitrate = float(np.mean(bitrates))
    switches = int(np.sum(np.diff(bitrates) != 0))

    qoe = (
        avg_bitrate
        - REBUFFER_PENALTY * (total_rebuffer / len(trace))
        - SMOOTHNESS_PENALTY * (total_variation / len(trace))
    )

    stats = {
        "Avg Bitrate (Mbps)": avg_bitrate,
        "Rebuffer (s)": total_rebuffer,
        "Switches": switches,
        "Variation (Mbps)": total_variation,
        "QoE": qoe,
    }

    return pd.DataFrame(rows), stats


print("Segment-level simulator ready")


## 步骤三：真正训练一个 Tabular Q-Learning

训练阶段使用大量随机生成的带宽轨迹。

每一步：

**State → ε-greedy Action → Reward → Q(s,a) Update**

Q-Learning 更新公式：

\[
Q(s,a)\leftarrow Q(s,a)+\alpha[r+\gamma\max_{a'}Q(s',a')-Q(s,a)]
\]

这里特别区分两个阶段：

- **训练阶段**：保留 ε-greedy 随机探索；
- **测试阶段**：`epsilon = 0`，只使用学到的最优动作。

因此不会再出现“评测时仍随机探索、导致人为增加码率切换”的问题。


In [4]:
def train_q_learning(n_episodes=8000, seed=2026, gamma=0.9):
    rng = np.random.default_rng(seed)

    q_table = np.zeros((45, 3), dtype=float)
    visits = np.zeros((45, 3), dtype=int)
    episode_scores = []

    for episode in range(n_episodes):
        trace = generate_bandwidth_trace(60, rng)

        buffer = INITIAL_BUFFER
        prev_action = 1
        bw_estimate = INITIAL_BW_EST
        total_reward = 0.0

        epsilon = 0.5 * np.exp(-episode / (n_episodes * 0.35)) + 0.02

        for actual_bw in trace:
            state = state_index(buffer, bw_estimate, prev_action)

            if rng.random() < epsilon:
                action = int(rng.integers(0, 3))
            else:
                action = int(np.argmax(q_table[state]))

            new_buffer, rebuffer, variation, reward, download_time = environment_step(
                buffer, prev_action, action, actual_bw
            )

            next_bw_estimate = float(actual_bw)
            next_state = state_index(new_buffer, next_bw_estimate, action)

            visits[state, action] += 1
            alpha = max(0.03, 0.3 / (1 + 0.002 * visits[state, action]))

            td_target = reward + gamma * np.max(q_table[next_state])
            q_table[state, action] += alpha * (
                td_target - q_table[state, action]
            )

            total_reward += reward
            buffer = new_buffer
            prev_action = action
            bw_estimate = next_bw_estimate

        episode_scores.append(total_reward / len(trace))

    return q_table, np.array(episode_scores)


Q_TABLE, TRAINING_SCORES = train_q_learning()

def strategy_qlearning(buffer, prev_action, bw_estimate, segment_id):
    state = state_index(buffer, bw_estimate, prev_action)
    return int(np.argmax(Q_TABLE[state]))  # evaluation: epsilon = 0


plt.figure(figsize=(10, 4))
window = 200
moving_avg = np.convolve(
    TRAINING_SCORES,
    np.ones(window) / window,
    mode="valid"
)
plt.plot(np.arange(window - 1, len(TRAINING_SCORES)), moving_avg)
plt.xlabel("Training episode")
plt.ylabel("Mean reward per segment")
plt.title("Q-Learning Training Curve (200-episode moving average)")
plt.grid(alpha=0.2)
plt.show()

# Show a compact learned policy slice: previous bitrate = 5 Mbps
policy_rows = []
buffer_labels = ["0–4", "4–8", "8–12", "12–16", "16+"]
bw_labels = ["<4", "4–8", "8+"]

for buffer_state, buffer_label in enumerate(buffer_labels):
    row = {"Buffer (s)": buffer_label}
    representative_buffer = buffer_state * 4 + 1
    for bw_state, bw_label in enumerate(bw_labels):
        representative_bw = [2.5, 6.0, 10.0][bw_state]
        state = state_index(representative_buffer, representative_bw, 1)
        action = int(np.argmax(Q_TABLE[state]))
        row[f"BW {bw_label}"] = f"{QUALITY_LEVELS[action]:g} Mbps"
    policy_rows.append(row)

policy_table = pd.DataFrame(policy_rows)
print("Learned policy slice (previous bitrate = 5 Mbps):")
display(policy_table)


## 步骤四：统一评测三种策略

现在三种策略都在**同一条从未参与训练的测试带宽轨迹**上运行。

我们重点看四个结果：

- **Avg Bitrate**：平均画质；
- **Rebuffer**：累计卡顿时间；
- **Switches**：码率切换次数；
- **QoE**：综合体验得分。

> 重要原则：**不预设“AI 必须第一”。实验结果是什么，就解释什么。**


In [5]:
policies = {
    "Fixed High": strategy_fixed,
    "Buffer Heuristic": strategy_heuristic,
    "Q-Learning": strategy_qlearning,
}

histories = {}
results = []

for name, policy in policies.items():
    history, stats = simulate(test_bandwidth, policy)
    histories[name] = history
    results.append({"Strategy": name, **stats})

results_df = pd.DataFrame(results)

display(
    results_df.style.format({
        "Avg Bitrate (Mbps)": "{:.2f}",
        "Rebuffer (s)": "{:.2f}",
        "Variation (Mbps)": "{:.1f}",
        "QoE": "{:.2f}",
    })
)

best_strategy = results_df.loc[results_df["QoE"].idxmax(), "Strategy"]

print(f"Best QoE on this fixed test trace: {best_strategy}")
print(
    "Evaluation uses the learned Q-table deterministically: "
    "no random exploration is used."
)


## 步骤五：观察每种策略的时间过程

每张图同时画出：

- 实线：播放器选择的码率；
- 另一条实线：Buffer 水位；
- `x` 标记：发生 Rebuffer 的 Segment。

由于三种策略使用同一条测试网络轨迹，差异直接来自决策方法。


In [6]:
for name, history in histories.items():
    plt.figure(figsize=(12, 4))

    plt.step(
        history["video_time"],
        history["bitrate"],
        where="post",
        linewidth=2,
        label="Selected bitrate (Mbps)"
    )
    plt.plot(
        history["video_time"],
        history["buffer_after"],
        linewidth=1.8,
        label="Buffer after download (s)"
    )

    stalled = history["rebuffer"] > 0
    if stalled.any():
        plt.scatter(
            history.loc[stalled, "video_time"],
            np.zeros(stalled.sum()),
            marker="x",
            s=60,
            label="Rebuffer event"
        )

    row = results_df[results_df["Strategy"] == name].iloc[0]
    plt.title(
        f"{name} | Avg {row['Avg Bitrate (Mbps)']:.2f} Mbps | "
        f"Rebuffer {row['Rebuffer (s)']:.2f} s | QoE {row['QoE']:.2f}"
    )
    plt.xlabel("Video time (s)")
    plt.ylabel("Mbps / Buffer seconds")
    plt.ylim(-0.5, 20.5)
    plt.grid(alpha=0.2)
    plt.legend()
    plt.show()


## 实验结果与分析

在本 Notebook 的固定随机种子下，可以观察到一个非常典型的 ABR 权衡：

- **Fixed High** 始终保持最高码率，但当网络无法持续支撑 10 Mbps 时，会积累大量 Rebuffering，因此综合 QoE 最差。
- **Buffer Heuristic** 非常稳健，能够避免卡顿，但策略较保守，平均码率偏低。
- **Q-Learning** 在训练阶段从大量不同带宽轨迹中学习；测试时关闭探索。它不需要被人为保证“每项指标都最好”，而是直接优化长期 Reward，在本测试轨迹上取得更高的综合 QoE。

一个重要的课堂结论是：

> **学习算法并不会天然优于成熟规则。是否更好，取决于状态设计、奖励函数、训练分布以及测试网络条件。**

---

## 从实验到真实系统

真实播放器的 ABR 决策会更加复杂，通常会同时利用吞吐量估计、Buffer、历史下载表现、设备能力和播放约束。

- **dash.js** 提供 throughput-based 与 buffer-based（BOLA）等多种 ABR 规则，并可动态组合这些规则。
- **Pensieve** 是 MIT CSAIL 在 SIGCOMM 2017 提出的经典强化学习 ABR 研究系统，不是 Netflix 产品。它展示了用强化学习直接学习 ABR 策略的研究路径。

参考：

- [dash.js · Adaptive Bitrate Streaming](https://dashif.org/dash.js/pages/usage/abr/index.html)
- [MIT Pensieve · Neural Adaptive Video Streaming with Pensieve](https://web.mit.edu/pensieve/)

---

## 实验局限性

本实验有意保持小而清晰，因此仍有明显简化：

1. 使用仿真带宽，而不是真实移动网络 trace；
2. 只有 3 档码率和固定 2 秒 Segment；
3. Q-Learning 状态是离散的，仅 45 个状态；
4. 使用上一段实测吞吐量作为下一段带宽估计，没有复杂预测器；
5. Reward 是教学版 QoE，不代表任何平台的真实商业指标。

这些限制并不是缺陷，而是把复杂 ABR 问题压缩成一个 **12 分钟可以完整理解并运行的强化学习实验**。

---

## 思考与拓展

1. 把测试带宽整体调低，哪种策略退化最快？
2. 增大 `REBUFFER_PENALTY`，学到的策略会不会更保守？
3. 增大 `SMOOTHNESS_PENALTY`，码率切换是否减少？
4. 将状态中的 Bandwidth 从 3 档改成 5 档，是否一定更好？
5. 如果训练轨迹和测试轨迹分布明显不同，Q-Learning 是否还能保持优势？
6. 下一步能否把 Tabular Q-Learning 替换成 DQN / PPO，并使用真实网络 trace？


---

← [实验一：AI 驱动的网络流量分类](https://www.kaggle.com/code/guopingtan/fmi-demo1-traffic-classification)
&nbsp;|&nbsp;
🏠 [课程主页 · Course Home](https://www.kaggle.com/code/guopingtan/fmi-course-kaggle-hands-on-lab-start-here)
&nbsp;|&nbsp;
[实验三：视频质量评估 PSNR vs SSIM →](https://www.kaggle.com/code/guopingtan/fmi-demo3-quality-assessment)

**FMI Course · Kaggle Hands-on Lab** &nbsp;|&nbsp; MV-AI Lab · Hohai University
